In [1]:
# !pip install thetaflow
# !pip install lifelines
# !pip install seaborn

In [1]:
import warnings
import time

import pandas as pd
import numpy as np

import pickle

from matplotlib import pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import lifelines

import os

import tensorflow as tf
import tensorflow_probability as tfp

from tensorflow import keras
from tensorflow.keras import optimizers, initializers, regularizers, layers

from scipy.stats import norm, t
from scipy.special import gamma

import thetaflow as thf

2026-03-11 04:30:28.371481: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/natan/codes/thetaflow_method_paper/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1773214230.949941  205534 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3556 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1660, pci bus id: 0000:01:00.0, compute capability: 7.5


# The dataset

We will be considering a RNA-sequence count table as input to our neural network. This high dimensional table consists of a $n \times p$ matrix where $n$ is the number of patients and $p$ is the number of genes being studied. Each entry to this table represents the number pf times a specific gene appeared in the first tumor cell of a given breast cancer patient. With that, considering a neural network, we want to directly map that profile to a Weibull survival time distribution.

In [3]:
weibull_parameters = {
    "lam": {"link": tf.math.exp, "link_inv": tf.math.log, "par_type": "nn", "shape": 1, "init": 1.0}, # scale
    "rho": {"link": tf.math.exp, "link_inv": tf.math.log, "par_type": "nn", "shape": 1, "init": 1.0} # shape
}

def weibull_survival_loss(model, nn_output, data):
    # Unpack the tuple to include the censoring indicator
    X, y, delta = data
    
    # Extract the strictly positive parameters mapped by the neural network
    lam = model.get_variable("lam", nn_output)
    rho = model.get_variable("rho", nn_output)
    
    # Add a tiny epsilon to y to prevent log(0) exploding gradients 
    # if any survival times are exactly 0 in the dataset
    y_safe = y + 1e-8

    # 1. Calculate the log-hazard component (only applied to uncensored data via delta)
    log_hazard = tf.math.log(rho) + (rho - 1.0) * tf.math.log(y_safe) - rho * tf.math.log(lam)
    
    # 2. Calculate the cumulative hazard component (applied to all data)
    cum_hazard = tf.math.pow(y_safe / lam, rho)
    
    # 3. Combine to form the simplified log-likelihood
    loglik = tf.reduce_sum(delta * log_hazard - cum_hazard)

    # Return the negative log-likelihood for the optimizer to minimize
    return -loglik

def bct_neural_network(model, seed=None):
    initializer = initializers.GlorotNormal(seed = seed)
    
    # ----- Elastic Net Regularization -----
    # Crucial because p >> n (~60,000 genes vs ~1,000 patients).
    # L1: LASSO Sparsity - feature selection for genes
    # L2: Highly correlated gene clusters, which may lead to multicolinearity problems
    elastic_net = regularizers.L1L2(l1 = 1.0e-4, l2 = 1.0e-4)

    # ----- Compressor layer -----
    # Funnels ~60k normalized gene inputs to a 128 latent space
    model.dense1 = layers.Dense(
        units=128, 
        activation="gelu", 
        kernel_initializer=initializer, 
        kernel_regularizer=elastic_net,
        dtype=tf.float32, 
        name="gene_compressor"
    )
    
    # ----- Dropout Layer -----
    # Prevents the network from memorizing the small number of patients.
    model.dropout = layers.Dropout(rate = 0.5, seed = seed)

    # ----- Hidden dense layer -----
    model.dense2 = layers.Dense(
        units = 32, 
        activation = "gelu", 
        kernel_initializer = initializer, 
        dtype = tf.float32, 
        name = "latent_representation"
    )

    # ----- Output layer (Weibull parameters) -----
    # 2 outputs:
    # - shape parameter rho
    # - scale parameter lambda
    model.output_layer = layers.Dense(
        units = 2, 
        activation = None,
        kernel_initializer = initializer, 
        dtype = tf.float32, 
        name = "weibull_params"
    )
    
def weibull_network_call(model, x_input, training = False):
    # x_input shape: (batch_size, p_genes)
    x = model.dense1(x_input)
    # Dropout is typically only applied during the training phase
    x = model.dropout(x, training = training)
    x = model.dense2(x)
    return model.output_layer(x)

def weibull_network_call_nolast(model, x_input):
    # Pass through the first dense layer
    x = model.dense1(x_input)
    
    # Explicitly turn OFF dropout. 
    # This ensures deterministic feature extraction for the LLLA covariance.
    x = model.dropout(x, training = False) 
    
    # Pass through the final hidden layer
    x = model.dense2(x)
    
    # Return the deterministic latent representation phi(x)
    return x